<a href="https://colab.research.google.com/github/PranshuSinglaNITD/Breast-Cancer-type-detection-using-Pytorch/blob/main/convNeXt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import files

print("Please upload your kaggle.json file...")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\nAuthentication successful! Downloading dataset...")

!kaggle datasets download -d sabahesaraki/breast-ultrasound-images-dataset

print("\nDownload complete! Unzipping...")

# Unzip the dataset quietly (-q) and remove the zip file to save Colab space
!unzip -q breast-ultrasound-images-dataset.zip
!rm breast-ultrasound-images-dataset.zip

print("\nData is ready to use!")

Please upload your kaggle.json file...


Saving kaggle (3).json to kaggle (3).json
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory

Authentication successful! Downloading dataset...
Dataset URL: https://www.kaggle.com/datasets/sabahesaraki/breast-ultrasound-images-dataset
License(s): unknown
100% 195M/195M [00:01<00:00, 123MB/s]


Download complete! Unzipping...

Data is ready to use!


In [ ]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.transforms import v2
from torch.utils.data import DataLoader, Subset
import numpy as np

# 1. Define Transforms
train_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=[-15, 15]),
    v2.ColorJitter(brightness=0.2, contrast=0.1),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Point to the unzipped Kaggle folder
data_dir = '/content/Dataset_BUSI_with_GT'

# 3. Load folders with different transforms
train_dataset_full = datasets.ImageFolder(root=data_dir, transform=train_transforms)
test_dataset_full  = datasets.ImageFolder(root=data_dir, transform=val_transforms)

# 4. Calculate 80/20 Split
dataset_size = len(train_dataset_full)
test_size = int(0.2 * dataset_size)
train_size = dataset_size - test_size

generator = torch.Generator().manual_seed(42)
indices = torch.randperm(dataset_size, generator=generator).tolist()

train_indices = indices[:-test_size]
test_indices = indices[-test_size:]

# 5. Create non-overlapping Subsets
train_dataset = Subset(train_dataset_full, train_indices)
test_dataset  = Subset(test_dataset_full, test_indices)

# 6. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Total Images: {dataset_size} | Training: {train_size} | Testing: {test_size}")
print(f"Classes found: {train_dataset_full.classes}")

Total Images: 1578 | Training: 1263 | Testing: 315
Classes found: ['benign', 'malignant', 'normal']


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 14.2 MB/s eta 0:00:00


In [ ]:
#optuna parameter
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Optuna tuning on: {device}")

# --- THE FIX: Calculate class weights right here before tuning starts ---
# (This assumes you have successfully run Cell 2 so train_indices is in memory)
train_targets = [train_dataset_full.targets[i] for i in train_indices]
class_weights_np = compute_class_weight('balanced', classes=np.unique(train_targets), y=train_targets)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)

def objective(trial):

    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.6)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-1, log=True)

    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_features = model.classifier[2].in_features

    model.classifier[2] = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(in_features, 3)
    )
    model = model.to(device)

    # Apply the settings
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    num_epochs = 5

    for epoch in range(num_epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    model.eval()
    correct_test, total_test = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    accuracy = correct_test / total_test
    return accuracy

print("Starting Hyperparameter Tuning with Optuna...")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("\n--- TUNING COMPLETE ---")
print("The absolute best hyperparameters found were:")
print(study.best_params)

[I 2026-03-21 11:02:27,307] A new study created in memory with name: no-name-addb8e9e-5ca4-4c8b-a0e5-f4f3034406d1


Running Optuna tuning on: cuda
Starting Hyperparameter Tuning with Optuna...


[I 2026-03-21 11:04:33,742] Trial 0 finished with value: 0.9079365079365079 and parameters: {'dropout_rate': 0.18295802399046815, 'lr': 0.0004358105044860001, 'weight_decay': 0.00040844096111967586}. Best is trial 0 with value: 0.9079365079365079.
[I 2026-03-21 11:06:34,186] Trial 1 finished with value: 0.5841269841269842 and parameters: {'dropout_rate': 0.3950843275796133, 'lr': 0.0017288308603987644, 'weight_decay': 0.00010308034363431376}. Best is trial 0 with value: 0.9079365079365079.
[I 2026-03-21 11:08:35,540] Trial 2 finished with value: 0.9365079365079365 and parameters: {'dropout_rate': 0.3042772099136617, 'lr': 3.163815062965008e-05, 'weight_decay': 0.009740979568942789}. Best is trial 2 with value: 0.9365079365079365.
[I 2026-03-21 11:10:36,611] Trial 3 finished with value: 0.5841269841269842 and parameters: {'dropout_rate': 0.3004337524536034, 'lr': 0.002451343987688368, 'weight_decay': 6.627421375302279e-05}. Best is trial 2 with value: 0.9365079365079365.
[I 2026-03-21 1


--- TUNING COMPLETE ---
The absolute best hyperparameters found were:
{'dropout_rate': 0.5336203737469998, 'lr': 0.0001775491855875906, 'weight_decay': 1.1327410798054477e-05}


In [ ]:

import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Final Model on: {device}")

#those from optuna
OPTUNA_DROPOUT = 0.5336
OPTUNA_LR = 1.775e-4       # 0.0001775
OPTUNA_WEIGHT_DECAY = 1.132e-5 # 0.00001132

mammo_model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
in_features = mammo_model.classifier[2].in_features

mammo_model.classifier[2] = nn.Sequential(
    nn.Dropout(p=OPTUNA_DROPOUT),
    nn.Linear(in_features, 3)
)
mammo_model = mammo_model.to(device)

train_targets = [train_dataset_full.targets[i] for i in train_indices]
class_weights_np = compute_class_weight('balanced', classes=np.unique(train_targets), y=train_targets)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(mammo_model.parameters(), lr=OPTUNA_LR, weight_decay=OPTUNA_WEIGHT_DECAY)

num_epochs = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

best_val_acc = 0.0

for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} [LR: {scheduler.get_last_lr()[0]:.6f}] ---")

    # --- TRAINING ---
    mammo_model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = mammo_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_acc = 100 * correct_train / total_train
    print(f"Training Loss: {running_loss/len(train_loader):.4f} | Training Acc: {train_acc:.2f}%")

    # --- VALIDATION ---
    mammo_model.eval()
    correct_test, total_test = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = mammo_model(images)
            _, predicted = torch.max(outputs.data, 1)

            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_acc = 100 * correct_test / total_test
    print(f"Validation Acc: {test_acc:.2f}%")

    scheduler.step()

    # Save the absolute best model
    if test_acc > best_val_acc:
        best_val_acc = test_acc
        torch.save(mammo_model.state_dict(), 'convnext_best_mammography_model.pth')
print(f"\nFinal Training Complete! Best Validation Accuracy: {best_val_acc:.2f}%")

Training Final Model on: cuda

--- Epoch 1/30 [LR: 0.000178] ---
Training Loss: 0.5595 | Training Acc: 74.98%
Validation Acc: 90.79%
🌟 Peak accuracy reached! Model weights saved.

--- Epoch 2/30 [LR: 0.000177] ---
Training Loss: 0.3327 | Training Acc: 86.70%
Validation Acc: 92.06%
🌟 Peak accuracy reached! Model weights saved.

--- Epoch 3/30 [LR: 0.000176] ---
Training Loss: 0.3063 | Training Acc: 87.97%
Validation Acc: 92.06%

--- Epoch 4/30 [LR: 0.000173] ---
Training Loss: 0.2489 | Training Acc: 89.55%
Validation Acc: 92.06%

--- Epoch 5/30 [LR: 0.000170] ---
Training Loss: 0.2671 | Training Acc: 90.10%
Validation Acc: 89.52%

--- Epoch 6/30 [LR: 0.000166] ---
Training Loss: 0.1802 | Training Acc: 92.48%
Validation Acc: 93.02%
🌟 Peak accuracy reached! Model weights saved.

--- Epoch 7/30 [LR: 0.000161] ---
Training Loss: 0.1338 | Training Acc: 94.22%
Validation Acc: 92.70%

--- Epoch 8/30 [LR: 0.000155] ---
Training Loss: 0.1289 | Training Acc: 95.25%
Validation Acc: 93.33%
🌟 Peak a